# 02. Data Preprocessing

## Objective

This notebook prepares the raw credit-card default dataset for
feature engineering and downstream machine-learning tasks.

The preprocessing steps are based on the findings from the EDA notebook.

The main objectives are:

- Preserve the raw dataset as the source of truth
- Standardize column names
- Remove identifier columns
- Validate data quality
- Identify feature types
- Preserve the semantic meaning of categorical and repayment-status variables
- Prepare a clean dataset for feature engineering

In [12]:
import pandas as pd
import numpy as np

## 1. Load Raw Dataset

The raw dataset is loaded directly from the source `.xls` file.
The raw file is preserved unchanged and serves as the source of truth.

In [13]:
RAW_FILE = "../data/raw/default of credit card clients.xls"

df = pd.read_excel(RAW_FILE, header=1)

df.head()

,ID,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,...,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default payment next month
0,1,20000,2,2,1,24,2,2,-1,-1,...,0,0,0,0,689,0,0,0,0,1
1,2,120000,2,2,2,26,-1,2,0,0,...,3272,3455,3261,0,1000,1000,1000,0,2000,1
2,3,90000,2,2,2,34,0,0,0,0,...,14331,14948,15549,1518,1500,1000,1000,1000,5000,0
3,4,50000,2,2,1,37,0,0,0,0,...,28314,28959,29547,2000,2019,1200,1100,1069,1000,0
4,5,50000,1,2,1,57,-1,0,-1,0,...,20940,19146,19131,2000,36681,10000,9000,689,679,0


## 2. Standardize Column Names

Column names are standardized to a consistent format to make
subsequent preprocessing and feature engineering steps easier.

In [15]:
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace("/", "_")
    .str.replace("-", "_")
)

df.columns.tolist()

['id',
 'limit_bal',
 'sex',
 'education',
 'marriage',
 'age',
 'pay_0',
 'pay_2',
 'pay_3',
 'pay_4',
 'pay_5',
 'pay_6',
 'bill_amt1',
 'bill_amt2',
 'bill_amt3',
 'bill_amt4',
 'bill_amt5',
 'bill_amt6',
 'pay_amt1',
 'pay_amt2',
 'pay_amt3',
 'pay_amt4',
 'pay_amt5',
 'pay_amt6',
 'default_payment_next_month']

## 3. Define Target and Feature Groups

Based on the EDA findings, the variables are grouped according to
their semantic meaning and role in the dataset.

In [16]:
target = "default_payment_next_month"

categorical_cols = [
    "sex",
    "education",
    "marriage"
]

pay_status_cols = [
    "pay_0",
    "pay_2",
    "pay_3",
    "pay_4",
    "pay_5",
    "pay_6"
]

bill_cols = [
    "bill_amt1",
    "bill_amt2",
    "bill_amt3",
    "bill_amt4",
    "bill_amt5",
    "bill_amt6"
]

payment_cols = [
    "pay_amt1",
    "pay_amt2",
    "pay_amt3",
    "pay_amt4",
    "pay_amt5",
    "pay_amt6"
]

numerical_cols = [
    "limit_bal",
    "age",
    *bill_cols,
    *payment_cols
]

## 4. Remove Identifier

The `ID` column is an identifier and does not represent a meaningful
customer characteristic for prediction or feature engineering.
Therefore, it is removed from the feature set.

In [17]:
df = df.drop(columns=["id"])

df.shape

(30000, 24)

## 5. Missing Value Validation

Missing values are checked before further preprocessing.
Although the EDA found no missing values, this validation is retained
as part of the preprocessing pipeline.

In [18]:
missing_values = df.isnull().sum()

missing_values[missing_values > 0]

Series([], dtype: int64)

## 6. Duplicate Row Validation

Duplicate records are checked to ensure that identical observations
are not unintentionally repeated in the dataset.

In [19]:
duplicate_count = df.duplicated().sum()

duplicate_count

np.int64(35)

In [22]:
## 6.1 Investigate Duplicate Feature Rows

feature_cols = [col for col in df.columns if col != target]

duplicate_mask = df.duplicated(subset=feature_cols, keep=False)

duplicate_groups = (
    df.loc[duplicate_mask]
    .groupby(feature_cols)[target]
    .nunique()
)

print("Rows involved in duplicate feature groups:", duplicate_mask.sum())
print("Number of duplicate feature groups:", duplicate_groups.size)
print("Groups with conflicting target labels:",
      (duplicate_groups > 1).sum())


Rows involved in duplicate feature groups: 108
Number of duplicate feature groups: 52
Groups with conflicting target labels: 21


### Duplicate Investigation

After removing the identifier column, 108 rows were found to belong
to 52 groups with identical feature values.

Among these groups, 21 contained conflicting target labels.

These observations were retained because identical feature values can
legitimately correspond to different outcomes, and removing them could
discard valid observations or introduce bias into the dataset.

## 7. Data Type Validation

The dataset is checked to ensure that numerical, categorical, repayment-status,
and target variables have appropriate data types before feature engineering.

Categorical and repayment-status variables are kept in their original numeric
representation at this stage so that their semantic meaning is preserved.

In [23]:
df.dtypes

limit_bal                     int64
sex                           int64
education                     int64
marriage                      int64
age                           int64
pay_0                         int64
pay_2                         int64
pay_3                         int64
pay_4                         int64
pay_5                         int64
pay_6                         int64
bill_amt1                     int64
bill_amt2                     int64
bill_amt3                     int64
bill_amt4                     int64
bill_amt5                     int64
bill_amt6                     int64
pay_amt1                      int64
pay_amt2                      int64
pay_amt3                      int64
pay_amt4                      int64
pay_amt5                      int64
pay_amt6                      int64
default_payment_next_month    int64
dtype: object

## 8. Validate Feature Groups

The predefined feature groups are checked against the dataset columns
to ensure that no expected variables are missing or incorrectly assigned.

In [24]:
expected_cols = (
    categorical_cols
    + pay_status_cols
    + bill_cols
    + payment_cols
    + ["limit_bal", "age"]
    + [target]
)

missing_from_groups = set(expected_cols) - set(df.columns)
extra_columns = set(df.columns) - set(expected_cols)

print("Missing from feature groups:", missing_from_groups)
print("Unexpected columns:", extra_columns)

Missing from feature groups: set()
Unexpected columns: set()


## 9. Validate Categorical and Repayment-Status Values

The categorical and repayment-status variables are inspected for
unexpected values before feature engineering.

The variables are retained in their original numeric representation,
while their semantic meaning is preserved for later encoding or
feature construction.

In [25]:
for col in categorical_cols + pay_status_cols:
    print(f"\n{col}:")
    print(sorted(df[col].unique()))


sex:
[np.int64(1), np.int64(2)]

education:
[np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6)]

marriage:
[np.int64(0), np.int64(1), np.int64(2), np.int64(3)]

pay_0:
[np.int64(-2), np.int64(-1), np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8)]

pay_2:
[np.int64(-2), np.int64(-1), np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8)]

pay_3:
[np.int64(-2), np.int64(-1), np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8)]

pay_4:
[np.int64(-2), np.int64(-1), np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8)]

pay_5:
[np.int64(-2), np.int64(-1), np.int64(0), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8)]

pay_6:
[np.int64(-2), np.int64(-1), np.in

## 10. Validate Numerical Feature Ranges

The numerical variables are checked for invalid or physically
impossible values before feature engineering.

Extreme but valid financial observations are retained because they
may represent genuine customer behavior and are relevant for
downstream monitoring.

In [26]:
range_checks = {
    "limit_bal": df["limit_bal"].min(),
    "age": df["age"].min(),
    "bill_amount_min": df[bill_cols].min().min(),
    "payment_amount_min": df[payment_cols].min().min()
}

range_checks

{'limit_bal': np.int64(10000),
 'age': np.int64(21),
 'bill_amount_min': np.int64(-339603),
 'payment_amount_min': np.int64(0)}

## 11. Validate Target Values

The target variable is checked to ensure that it contains only the
expected binary class labels.

In [27]:
print("Target values:", sorted(df[target].unique()))
print("\nTarget counts:")
print(df[target].value_counts())

Target values: [np.int64(0), np.int64(1)]

Target counts:
default_payment_next_month
0    23364
1     6636
Name: count, dtype: int64


## 12. Create Cleaned Dataset

The validated dataset is copied as the cleaned dataset for downstream
feature engineering.

No observations are removed at this stage because the validation
process did not identify any invalid records requiring deletion.

In [28]:
cleaned_df = df.copy()

print("Cleaned dataset shape:", cleaned_df.shape)

Cleaned dataset shape: (30000, 24)


## 13. Save Cleaned Dataset

The validated dataset is saved as an intermediate cleaned dataset.

The original raw dataset remains unchanged and serves as the source
of truth. The cleaned dataset will be used as the input for the
feature-engineering stage.

In [29]:
OUTPUT_FILE = "../data/preprocessed.csv"

cleaned_df.to_csv(OUTPUT_FILE, index=False)

print(f"Preprocessed dataset saved to: {OUTPUT_FILE}")

Preprocessed dataset saved to: ../data/preprocessed.csv
